In [1]:
from pathlib import Path
import pandas as pd
from catboost.utils import eval_metric
from narwhals import DataFrame

path_to_csv = Path("../datasets/rnc_dataset.csv")

In [2]:
try:
    df = pd.read_csv(path_to_csv)
except FileNotFoundError as e:
    print(f"Error: {e}, try to specify path correct.")

In [3]:
df = df.drop(
    columns=["itemId", "split", "targetDetectedMcIds", "targetSplitMcIds", "caseType"]
)

In [4]:
df.tail()

,sourceMcId,sourceMcTitle,description,shouldSplit
2995,107,Малярные работы,"Окраска стен в санузел. Делаю моющаяся краска,...",False
2996,101,Ремонт квартир и домов под ключ,Комплексный ремонт дома для дома. Покраска пот...,False
2997,111,Демонтажные работы,Демонтаж перегородок в студия. Делаю демонтаж ...,False
2998,101,Ремонт квартир и домов под ключ,Ремонт вторички под ключ в офисе. Делаем отдел...,True
2999,101,Ремонт квартир и домов под ключ,Полная отделка квартиры в квартире. Выполняем ...,True


In [5]:
from utils.description_features_utils import (
    has_bull_markers,
    slash_counter,
    has_slash,
    paragraph_counter,
    word_separately_in_desc,
    count_the_occurrence_of_words_for_separation,
    turkney_count,
)


def features_from_description(df: DataFrame) -> dict:
    desc = df["description"]
    return dict(
        desc_words_count=len(desc.split()),
        desc_length=len(desc),
        has_bull_markers=has_bull_markers(desc),
        has_slashes=has_slash(desc),
        slash_counted=slash_counter(desc),
        paragraph_counted=paragraph_counter(desc),
        is_word_separately_included=word_separately_in_desc(desc),
        should_split_words_trigger_counter=count_the_occurrence_of_words_for_separation(
            desc
        ),
        turnkey_count=turkney_count(desc),
    )


features_desc = df.apply(features_from_description, axis=1)

df_with_desc_features = pd.concat((df, pd.DataFrame(features_desc.tolist())), axis=1)

In [6]:
print(
    f"Random should split string for repr: \n\n {df_with_desc_features.iloc[1].to_string()}"
)

Random should split string for repr: 

 sourceMcId                                                                          101
sourceMcTitle                                           Ремонт квартир и домов под ключ
description                           Косметический ремонт под ключ в доме. Отдельно...
shouldSplit                                                                        True
desc_words_count                                                                     34
desc_length                                                                         251
has_bull_markers                                                                  False
has_slashes                                                                       False
slash_counted                                                                         0
paragraph_counted                                                                     0
is_word_separately_included                                                     

In [7]:
from sklearn.model_selection import train_test_split

X = df_with_desc_features.drop("shouldSplit", axis=1)
y = df_with_desc_features["shouldSplit"]

cat_features = ["sourceMcTitle", "sourceMcId"]
text_features = ["description"]

X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, test_size=0.15, random_state=45, shuffle=True, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_holdout,
    y_holdout,
    test_size=0.5,
    random_state=52,
    shuffle=True,
    stratify=y_holdout,
)

In [9]:
from catboost import CatBoostClassifier

model = CatBoostClassifier(
    iterations=3000,
    learning_rate=0.03,
    loss_function="Logloss",
    eval_metric="AUC",
)

model.fit(
    X_train,
    y_train,
    eval_set=(X_val, y_val),
    early_stopping_rounds=50,
    use_best_model=True,
    cat_features=cat_features,
    text_features=text_features,
    verbose=50,
ак)

0:	test: 0.0996946	best: 0.0996946 (0)	total: 118ms	remaining: 5m 55s
50:	test: 0.9996606	best: 0.9996606 (41)	total: 7.09s	remaining: 6m 49s
100:	test: 0.9999152	best: 0.9999152 (88)	total: 11.1s	remaining: 5m 19s
150:	test: 1.0000000	best: 1.0000000 (105)	total: 14.3s	remaining: 4m 29s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 1
bestIteration = 105

Shrink model to first 106 iterations.


CatBoostClassifier(eval_metric='AUC', iterations=3000, learning_rate=0.03, loss_function='Logloss')

In [14]:
from sklearn.metrics import accuracy_score, confusion_matrix

y_pred = model.predict(X_val)

acc_score = accuracy_score(y_val, y_pred)
cfs_mtrx = confusion_matrix(y_val, y_pred)

In [16]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, log_loss, classification_report


def report_binary(model, X, y, name="data"):
      y = np.asarray(y).astype(int)
      y_pred = model.predict(X).astype(int)
      proba = model.predict_proba(X)[:, 1]
      print(f"=== {name} ===")
      print(f"accuracy:  {accuracy_score(y, y_pred):.4f}")
      print(f"precision: {precision_score(y, y_pred, zero_division=0):.4f}")
      print(f"recall:    {recall_score(y, y_pred, zero_division=0):.4f}")
      print(f"f1:        {f1_score(y, y_pred, zero_division=0):.4f}")
      print(f"roc_auc:   {roc_auc_score(y, proba):.4f}")
      print(f"log_loss:  {log_loss(y, proba):.4f}")
      cm = confusion_matrix(y, y_pred, labels=[0, 1])
      print("confusion_matrix [rows=true 0,1 | cols=pred 0,1]:")
      print(cm)
      print(classification_report(y, y_pred, labels=[0, 1], target_names=["no_split", "split"]))

In [18]:
print(f"Report for validation set: ---------\n\n{report_binary(model, X_val, y_val, name="validation")}")
print(f"Report for validation set: ---------\n\n{report_binary(model, X_test, y_test, name="validation")}")



=== validation ===
accuracy:  0.9911
precision: 1.0000
recall:    0.9759
f1:        0.9878
roc_auc:   1.0000
log_loss:  0.0336
confusion_matrix [rows=true 0,1 | cols=pred 0,1]:
[[142   0]
 [  2  81]]
              precision    recall  f1-score   support

    no_split       0.99      1.00      0.99       142
       split       1.00      0.98      0.99        83

    accuracy                           0.99       225
   macro avg       0.99      0.99      0.99       225
weighted avg       0.99      0.99      0.99       225

Report for validation set: ---------

None
=== validation ===
accuracy:  0.9867
precision: 1.0000
recall:    0.9643
f1:        0.9818
roc_auc:   0.9999
log_loss:  0.0439
confusion_matrix [rows=true 0,1 | cols=pred 0,1]:
[[141   0]
 [  3  81]]
              precision    recall  f1-score   support

    no_split       0.98      1.00      0.99       141
       split       1.00      0.96      0.98        84

    accuracy                           0.99       225
   macro avg